In [11]:
import h5py
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.optim as optim

# Custom Dataset for reading HDF5 data
class HDF5Dataset(Dataset):
    def __init__(self, hdf5_file_path):
        self.hdf5_file_path = hdf5_file_path
        with h5py.File(hdf5_file_path, 'r') as hf:
            self.num_files = hf['Layer_1'].shape[1]  # Number of audio files

    def __len__(self):
        return self.num_files

    def __getitem__(self, idx):
        with h5py.File(self.hdf5_file_path, 'r') as hf:
            X = hf['Layer_1'][:, idx, :]  # Shape: (315, 192)
            y = hf['LID'][idx].decode('utf-8')
        
        X = torch.tensor(X, dtype=torch.float32)
        label_map = {'English': 0, 'Mandarin': 1}  # Example mapping
        y = label_map[y]  # Directly map y to an integer
        return X, y

# Define your language identification model
class LanguageIdentificationModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(LanguageIdentificationModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        out = self.fc(hn[-1])  # Use the hidden state of the last LSTM layer
        return out

# Initialize dataset and dataloader
hdf5_file_path = '/export/c09/lavanya/languageIdentification/zinglish/small/embeddingSmall/embed_1727983789.h5'
dataset = HDF5Dataset(hdf5_file_path)

# Split the dataset into training and testing sets
train_size = int(0.8 * len(dataset))  # 80% for training
test_size = len(dataset) - train_size  # 20% for testing
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Create dataloaders for training and testing
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Model parameters
input_dim = 192  # Dimension of each embedding
hidden_dim = 128  # LSTM hidden layer size
num_classes = 2  # Number of languages (English, Mandarin, etc.)

# Instantiate the model, optimizer, and loss function
model = LanguageIdentificationModel(input_dim, hidden_dim, num_classes)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        X_batch, y_batch = batch  # Get a batch of embeddings and labels

        # Convert y_batch to a tensor of integers for CrossEntropyLoss
        y_batch = torch.tensor(y_batch, dtype=torch.long)

        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)  # Calculate the loss

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader)}')

# Testing loop to calculate accuracy
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        X_batch, y_batch = batch
        outputs = model(X_batch)
        _, predicted = torch.max(outputs.data, 1)  # Get the predicted class
        total += y_batch.size(0)  # Number of samples in the batch
        correct += (predicted == y_batch).sum().item()  # Count correct predictions

accuracy = 100 * correct / total
print(f'Accuracy of the model on the test set: {accuracy:.2f}%')

print("Training complete.")


/tmp/ipykernel_3753550/4134480037.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_batch = torch.tensor(y_batch, dtype=torch.long)


Epoch [1/10], Loss: 0.6800106565157572
Epoch [2/10], Loss: 0.6315661470095316
Epoch [3/10], Loss: 0.5520399610201517
Epoch [4/10], Loss: 0.416628897190094
Epoch [5/10], Loss: 0.17696169515450796
Epoch [6/10], Loss: 0.13862395410736403
Epoch [7/10], Loss: 0.1303852954879403
Epoch [8/10], Loss: 0.2143546019991239
Epoch [9/10], Loss: 0.6099347174167633
Epoch [10/10], Loss: 0.5699795186519623
Accuracy of the model on the test set: 85.00%
Training complete.
